# Vision(DL) 100건 분석 Notebook

## tl;dr

- Qwen3-VL-4B: JSON valid 394/400(98.5%), fallback 6/400(1.5%), 비사고 부정 0/400.
- 선택 VideoMAE checkpoint: 고정 test 40건 accuracy 25/40(62.5%), macro F1 62.06%.

## Context & Methods

동일한 400개 asset과 동일한 VideoMAE·YOLO·16프레임 근거로 Qwen2.5-VL-3B와 Qwen3-VL-4B를 paired 비교한다. VideoMAE는 32프레임, 설명 근거는 충돌 후보 중심 16프레임이다.

### Key Assumptions

- `vision_100_metrics.json`은 검증된 RunPod CSV/JSON/로그에서 집계된 compact snapshot이다.
- 전체 400건 VideoMAE 72%는 독립 test가 아니므로 일반화 성능으로 사용하지 않는다.

In [ ]:
import json
from pathlib import Path

candidates = [
    Path('docs/vision/vision_100_metrics.json'),
    Path('vision_100_metrics.json'),
]
metrics_path = next((path for path in candidates if path.is_file()), None)
if metrics_path is None:
    raise FileNotFoundError('Run from repository root or docs/vision.')
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
metrics['scope']


## Data

카테고리별 100건, 총 400건이며 train/validation/test는 280/80/40이다. asset_id, incident_id, SHA-256은 각각 400/400 고유하다.

In [ ]:
scope = metrics['scope']
assert sum(scope['categories'].values()) == 400
assert scope['unique_asset_ids'] == scope['total_videos']
assert scope['unique_incident_ids'] == scope['total_videos']
assert scope['unique_sha256'] == scope['total_videos']
print('dataset checks: passed')
print('split:', scope['split'])
print('evidence frames:', scope['total_evidence_frames'])


## Results

### VideoMAE checkpoint

In [ ]:
videomae = metrics['videomae_fixed_test_40']
for name, result in videomae.items():
    accuracy = result['accuracy']
    print(f"{name:30s} accuracy={accuracy['numerator']}/{accuracy['denominator']} ({accuracy['rate']:.1%}) macro_f1={result['macro_f1']:.2%}")
selected = videomae['old_per_label_300_selected']
assert selected['accuracy']['rate'] == max(item['accuracy']['rate'] for item in videomae.values())


### Qwen paired comparison

In [ ]:
q25 = metrics['qwen']['qwen25']['overall']
q3 = metrics['qwen']['qwen3']['overall']
comparisons = {
    'JSON valid (pp)': 100 * (q3['model_json_valid']['rate'] - q25['model_json_valid']['rate']),
    'fallback (pp)': 100 * (q3['fallback']['rate'] - q25['fallback']['rate']),
    'non-accident denial (pp)': 100 * (q3['non_accident_denial']['rate'] - q25['non_accident_denial']['rate']),
    'mean latency (sec)': q3['latency_seconds']['mean'] - q25['latency_seconds']['mean'],
}
for metric, delta in comparisons.items():
    print(f'{metric:28s} {delta:+.3f}')
assert q3['model_json_valid']['rate'] > q25['model_json_valid']['rate']
assert q3['non_accident_denial']['numerator'] == 0


In [ ]:
categories = metrics['qwen']['qwen3']['per_category']
print('category               valid fallback denial mean_latency')
for category, row in categories.items():
    print(f"{category:24s} {row['json_valid']:>5d} {row['fallback']:>8d} {row['denial']:>6d} {row['mean_latency']:>12.3f}")


## Takeaways

1. Qwen3-VL-4B를 기본 설명 모델로 사용한다.
2. VideoMAE 유형과 `confirmed_accident`는 읽기 전용이며 Qwen은 설명만 생성한다.
3. Qwen fallback과 human review는 유지한다.
4. 사람 설명 일치도와 직접 impact 근거는 아직 검증되지 않았다고 명시한다.